In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [187]:
df = pd.read_excel("GDP_structure_data/cash-crops.xlsx", header=None)

df.columns = df.iloc[5]
df = df[6:]

df.columns = ["Crop", "Type"] + list(df.columns[2:])

df["Crop"] = df["Crop"].ffill()
df["Type"] = df["Type"].astype(str).str.strip()
df = df[~df["Crop"].astype(str).str.contains(
    "Total|Source|Honey is included|Preliminary|अनुसूची|Productivity",
    na=False
)]
data = df.melt(id_vars=["Crop", "Type"], var_name="Year", value_name="Value")

data.tail(10)  ## there were some nan so i checked if it is real or something weirdo
data.isnull().sum()
data.head(50) ## checking if there are not any values

data["Value"]=data["Value"].replace("-","0")
data.info()

data["Year"] = data["Year"].str[:4].astype(int)
data["Year"] = pd.to_datetime(data["Year"], format="%Y")
data["Value"] = pd.to_numeric(data["Value"])

data.to_csv("cleanedData/clean_cashCrops.csv")


<class 'pandas.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Crop    117 non-null    str   
 1   Type    117 non-null    str   
 2   Year    117 non-null    str   
 3   Value   117 non-null    object
dtypes: object(1), str(3)
memory usage: 3.8+ KB


In [9]:
df = pd.read_excel("GDP_structure_data/food-crops.xlsx",header=None)
df.head(10)

df.columns = df.iloc[5]
df = df[6:]

df.columns = ["Crop", "Type"] + list(df.columns[2:])
df["Crop"] = df["Crop"].ffill()
df["Type"].unique()
df = df[~df["Crop"].astype(str).str.contains("Ministry of Agriculture|Preliminary Estimate|fiscal year|Total|Productivity")]
data =df.melt(id_vars=["Crop", "Type"], var_name="Year", value_name="Value")

data.isnull().sum()
data.info()

data["Year"] = data["Year"].str[:4].astype(int)
data["Year"] = pd.to_datetime(data["Year"], format="%Y")
data["Value"] = pd.to_numeric(data["Value"])

data.info()
data.to_csv("cleanedData/clean_foodCrops.csv")

<class 'pandas.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Crop    234 non-null    str   
 1   Type    234 non-null    str   
 2   Year    234 non-null    str   
 3   Value   234 non-null    object
dtypes: object(1), str(3)
memory usage: 7.4+ KB
<class 'pandas.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Crop    234 non-null    str           
 1   Type    234 non-null    str           
 2   Year    234 non-null    datetime64[us]
 3   Value   234 non-null    float64       
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 7.4 KB


In [21]:
df = pd.read_excel("GDP_structure_data/livestock-production.xlsx",header=None)
df.head(10)
years = df.iloc[2, 1:].tolist()
clean_data = []

i = 3  # start from first data row

while i < len(df):
    category = df.iloc[i, 0]
    
    # Skip empty rows
    if pd.isna(category):
        i += 1
        continue
    
    production = df.iloc[i, 1:].values
    growth = df.iloc[i+1, 1:].values
    
    for j in range(len(years)):
        clean_data.append({
            "Year": years[j],
            "Category": category,
            "Production": production[j],
            "Growth Rate": growth[j]
        })
    
    i += 2  # move to next pair

clean_df = pd.DataFrame(clean_data)
clean_df["Main Category"] = clean_df["Category"].apply(
    lambda x: "Milk" if "Milk" in str(x)
    else "Meat" if "Meat" in str(x)
    else "Livestock"
)
clean_df = clean_df[~clean_df["Category"].astype(str).str.contains("Ministry of Agriculture|previous year")]
clean_df.isnull().sum()
clean_df.duplicated().sum()

clean_df["Production"] = pd.to_numeric(clean_df["Production"], errors="coerce")
clean_df["Growth Rate"] = pd.to_numeric(clean_df["Growth Rate"], errors="coerce")
clean_df.info()

clean_df["Year"] = clean_df["Year"].str[:4].astype(int)
clean_df["Year"] = pd.to_datetime(clean_df["Year"], format="%Y")

clean_df.info()
clean_df.to_csv("cleanedData/liveStockProduction.csv")

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Year           140 non-null    str    
 1   Category       140 non-null    str    
 2   Production     140 non-null    float64
 3   Growth Rate    140 non-null    float64
 4   Main Category  140 non-null    str    
dtypes: float64(2), str(3)
memory usage: 5.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Year           140 non-null    datetime64[us]
 1   Category       140 non-null    str           
 2   Production     140 non-null    float64       
 3   Growth Rate    140 non-null    float64       
 4   Main Category  140 non-null    str           
dtypes: datetime64[us](1), float64(2), str(2)
memory usage: 5.6 KB


In [22]:
clean_df.groupby("Category")["Growth Rate"].mean().sort_values(ascending=False)

Category
Chicken                             31.185430
Ducks                               22.248613
Cow                                  9.279267
Duck                                 7.819375
Pigs                                 7.700837
Eggs( In thousand)                   7.263469
Hen                                  7.136592
Net Meat Production(Metric Tons)     5.380476
Milk Production (Metric Tons)        4.707391
Goats                                3.943125
Sheep                                2.115514
Buffalo                              0.886930
Wool (kg)                           -3.387630
Name: Growth Rate, dtype: float64

In [44]:
df = pd.read_excel("GDP_structure_data/quarterly_GDP-2010.xlsx",header=None)
year_nep = df.iloc[0]
year_eng = df.iloc[1].ffill()
quarter = df.iloc[2]
columns = []

for i in range(len(df.columns)):
    if i == 0:
        columns.append("Code")
    elif i == 1:
        columns.append("Sector")
    else:
        y = year_eng[i]
        q = quarter[i]
        
        if pd.notna(y) and pd.notna(q):
            columns.append(f"{y}-{q}")
        else:
            columns.append(None)
df.columns = columns
df = df.iloc[3:].reset_index(drop=True)
df = df.loc[:, df.columns.notna()]
df_long = df.melt(
    id_vars=["Code", "Sector"],
    var_name="Year_Quarter",
    value_name="Value"
)
df_long['Value'] = pd.to_numeric(df_long['Value'])
df_long[['Year', 'Quarter']] = df_long['Year_Quarter'].str.split('-', expand=True)
quarter_map = {
    'Q1': 7,   # July
    'Q2': 10,  # October
    'Q3': 1,   # January
    'Q4': 4    # April
}

df_long['Start_Year'] = df_long['Year'].str[:4].astype(int)
df_long['Month'] = df_long['Quarter'].map(quarter_map)
df_long['Year_Adjusted'] = df_long['Start_Year']

df_long.loc[df_long['Quarter'].isin(['Q3', 'Q4']), 'Year_Adjusted'] += 1
df_long['Date'] = pd.to_datetime(
    dict(
        year=df_long['Year_Adjusted'],
        month=df_long['Month'],
        day=1
    )
)
df_long=df_long[~df_long.isnull()]
df_long.isnull().sum() # we have some null values after checking real data we can say that 
#this is due to unavailability of data in real dataset.so this is not by any cleaning issues
len(df_long[(df_long["Quarter"]=="Q3") & (df_long["Start_Year"]==2024) ])
len(df_long[(df_long["Quarter"]=="Q4") & (df_long["Start_Year"]==2024) ])

df_long=df_long[~df_long["Sector"].str.contains("Total")] # by this we cleaned null values that are in 'Code'.
df_long.isnull().sum()
df_long.duplicated().sum()
df_long.fillna(0) # we cant eliminate all data
df_long.to_csv("cleanedData/quarterly_gdp.csv")
df

,Code,Sector,2010/11-Q1,2010/11-Q2,2010/11-Q3,2010/11-Q4,2011/12-Q1,2011/12-Q2,2011/12-Q3,2011/12-Q4,...,2022/23-Q3,2022/23-Q4,2023/24-Q1,2023/24-Q2,2023/24-Q3,2023/24-Q4,2024/25-Q1,2024/25-Q2,2024/25-Q3,2024/25-Q4
0,A,"Agriculture, forestry and fishing",116306,129228,119720,115036,122157,122783,127420,133405,...,171299,169855,171693,175334,176171,178218,177380,181971,NaN,NaN
1,B,Mining and quarrying,2423,2260,1925,1908,2057,2293,2357,2242,...,4359,4442,4281,4037,3953,5115,4336,4434,NaN,NaN
2,C,Manufacturing,21605,20900,20500,21138,21620,22758,23735,24538,...,32617,32506,31747,30766,31494,32387,32644,33169,NaN,NaN
3,D,"Electricity, gas, steam and air conditioning s...",3378,3571,3614,3830,4057,4216,4297,3996,...,10489,17656,15044,15657,15956,16873,18092,17231,NaN,NaN
4,E,"Water supply; sewerage, waste management",2285,2291,2263,2304,2417,2471,2549,2613,...,4327,4374,4317,4399,4418,4420,4385,4421,NaN,NaN
5,F,Construction,26227,24310,21081,21002,21459,23856,24599,22890,...,41169,46895,41341,39504,38482,42384,41792,43108,NaN,NaN
6,G,Wholesale and retail trade; repair of motor ve...,54615,56958,55352,53199,53948,57447,56874,58403,...,87894,89913,85929,83935,86363,89976,87286,90217,NaN,NaN
7,H,Transportation and storage,18564,19466,19775,19368,20648,20773,20540,20448,...,30057,32182,34099,35175,35515,35390,37909,40319,NaN,NaN
8,I,Accommodation and food service activities,5661,6067,6354,6333,6280,6443,6437,6750,...,8824,9995,10524,10755,11029,11151,11354,11044,NaN,NaN
9,J,Information and communication,6661,7681,8450,8642,8978,9835,10548,10712,...,22280,23010,23842,23798,23750,22673,24338,24966,NaN,NaN


In [45]:
df = pd.read_excel("GDP_structure_data/national-sub-cpi.xlsx",header=None)
df = df.iloc[4:].reset_index(drop=True)
df = df.ffill(axis=1)
header = df.iloc[0:4]

new_columns = header.apply(
    lambda col: ' '.join([str(x) for x in col if pd.notna(x)]),
    axis=0
)
df.columns = new_columns

df.isnull().sum()
df.duplicated().sum()

df = df.iloc[4:].reset_index(drop=True)
df.columns = df.columns.str.strip()  # remove extra spaces
df = df.rename(columns={"Fiscal Year Quarter/ Month":"Fiscal Year"})
df['Fiscal Year'] = df['Fiscal Year'].astype(str)
df['Year'] = df['Fiscal Year'].str.split('/').str[0]
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-07-01')
df = df.set_index('Date')
df.info()
df.to_csv("cleanedData/CleanedNational-cpi.csv")
df

<class 'pandas.DataFrame'>
DatetimeIndex: 18 entries, 2006-07-01 to 2023-07-01
Data columns (total 28 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Fiscal Year                                18 non-null     str   
 1   Fiscal Year Overall Index                  18 non-null     object
 2   Fiscal Year Food and Beverages             18 non-null     object
 3   Cereal Grains & Their Products             18 non-null     object
 4   Cereal Grains & Pulses and Legumes         18 non-null     object
 5   Cereal Grains & Pulses and Vegetables      18 non-null     object
 6   Cereal Grains & Meat & Fish                18 non-null     object
 7   Cereal Milk Products and Eggs              18 non-null     object
 8   Cereal Milk Ghee and Oil                   18 non-null     object
 9   Cereal Milk Ghee Fruit                     18 non-null     object
 10  Cereal Sugar & Sugar  Products 

,Fiscal Year,Fiscal Year Overall Index,Fiscal Year Food and Beverages,Cereal Grains & Their Products,Cereal Grains & Pulses and Legumes,Cereal Grains & Pulses and Vegetables,Cereal Grains & Meat & Fish,Cereal Milk Products and Eggs,Cereal Milk Ghee and Oil,Cereal Milk Ghee Fruit,...,Cereal Fiscal Year Clothes & Footware,Cereal Fiscal Year Housing & Utilities,Furnishing & Household Equipment,Furnishing & Household Health,Furnishing & Transpor- tation,Furnishing & Commu- nication,Furnishing & Recreation & Culture,Furnishing & Recreation Education,Furnishing Miscellaneous Goods & Services,Year
Date,,,,,,,,,,,,,,,,,,,,,
2006-07-01,2006/07,49.83541,40.871883,45.156547,43.817651,30.623084,36.936272,42.30203,55.09575,35.760144,...,46.558639,62.858521,49.035172,66.279654,58.409705,123.456407,64.774625,53.633583,54.94305,2006
2007-07-01,2007/08,53.176594,44.691685,51.698512,49.757128,32.930114,39.846823,45.601352,66.632232,37.276833,...,48.133086,66.489553,51.884451,70.019168,59.753074,123.456407,67.547079,56.18995,55.969518,2007
2008-07-01,2008/09,59.858962,52.445883,59.302483,61.897568,36.637842,49.132865,52.278553,77.599649,43.276178,...,52.181663,72.025389,58.731912,73.500784,69.694008,123.579863,72.209844,60.901685,62.668573,2008
2009-07-01,2009/10,65.600152,60.391072,65.292204,77.99766,44.135691,59.42374,58.523699,74.030065,52.14038,...,56.140272,74.346868,62.362445,75.95081,66.416187,123.579863,77.502712,67.818913,67.368716,2009
2010-07-01,2010/11,71.85899,69.329409,74.382986,72.095538,59.598289,64.551853,67.046949,75.892456,62.251637,...,63.607648,79.823179,65.901065,79.110054,73.133034,110.493484,75.675413,71.528152,71.366539,2010
2011-07-01,2011/12,77.835475,74.562538,74.637868,72.058183,74.044694,69.402771,76.159364,86.859873,72.801048,...,73.009344,84.704238,74.724639,82.720619,84.524807,101.481166,81.535374,78.495505,78.443765,2011
2012-07-01,2012/13,85.506081,81.743766,81.392233,81.060786,78.27425,79.41645,82.718732,98.810219,77.351113,...,81.87123,93.692531,84.651159,88.329889,93.713454,99.382407,88.151459,88.019225,86.331362,2012
2013-07-01,2013/14,93.270805,91.216875,90.440535,85.020438,94.313605,93.865255,88.649657,99.741415,87.934228,...,90.958037,98.45454,92.417742,94.841801,98.710789,99.876233,94.137441,94.735954,92.598219,2013
2014-07-01,2014/15,100.000232,100.00242,99.9986,99.999873,99.998787,99.998201,100.000898,100.00008,100.000328,...,99.99986,100.002193,100.000501,99.99975,100.000423,99.999689,99.997402,99.999063,99.999593,2014


In [ ]:
value = [x for x in users if x%2 ==0 ]